# 05 — Executive summary

**Full A/B cycle: power planning → pre-registration → inference →
business significance → robustness.** This page restates the
headline numbers and the final decision; the details live in
notebooks 01–04 and the robustness check in notebook 06.

**Status: SUMMARY / DECISION.** Reviews the confirmatory and
robustness evidence together and states the final verdict.


In [1]:
import sys, pathlib
root = pathlib.Path.cwd()
while root != root.parent and not (root / 'data' / 'marketing_AB.csv').exists():
    root = root.parent
if str(root) not in sys.path: sys.path.insert(0, str(root))
from src.eda import load_marketing_ab, conversion_rates
from src.inference import two_proportion_ztest, wald_ci_difference, sample_size_proportions
df = load_marketing_ab(root / 'data' / 'marketing_AB.csv')
cr = conversion_rates(df).set_index('test_group')
x_ad, n_ad = cr.loc['ad', 'converted'], cr.loc['ad', 'users']
x_psa, n_psa = cr.loc['psa', 'converted'], cr.loc['psa', 'users']
p_ad, p_psa = x_ad / n_ad, x_psa / n_psa
z, pval = two_proportion_ztest(x_ad, n_ad, x_psa, n_psa)
lo, hi = wald_ci_difference(p_ad, p_psa, n_ad, n_psa)
n_req = sample_size_proportions(0.0179, 0.005)
import pandas as pd
rows = [
    ('Required n per group (MDE +0.5 pp, 80% power)', f'{n_req:,}'),
    ('psa group size (binding constraint)', '23,524'),
    ('Observed conversion rate ad vs psa', '2.55% vs 1.79%'),
    ('Difference (pp)', '+0.77'),
    ('Relative uplift', '+43.1%'),
    ('z statistic / two-sided p-value', f'{z:.2f} / {pval:.1e}'),
    ('95% CI for difference (pp)', '[+0.60, +0.94]'),
    ('Additional converters (95% CI)', '[3,500, 5,548]'),
]
display(pd.DataFrame(rows, columns=['Item', 'Value']).style.hide(axis='index'))

Item,Value
"Required n per group (MDE +0.5 pp, 80% power)","12,547"
psa group size (binding constraint),"23,524"
Observed conversion rate ad vs psa,2.55% vs 1.79%
Difference (pp),+0.77
Relative uplift,+43.1%
z statistic / two-sided p-value,7.37 / 1.7e-13
95% CI for difference (pp),"[+0.60, +0.94]"
Additional converters (95% CI),"[3,500, 5,548]"


## Final decision

Per the pre-registered rule (reports/pre_registration.md):
p < 0.05 **and** lower CI bound (0.60 pp) ≥ MDE (0.50 pp) — the
pooled difference is statistically significant on its own.

**But the pooled result is not uniform (see notebook 06):** the ad
advantage concentrates in high-exposure users (21+ ads, especially
50+, where 63% of ad conversions live). Among users with 1–20 ads
the difference is not significant. Removing the 50+ stratum drops
the pooled difference from +0.77 to about +0.15 pp (not
significant).

**Verdict: do not adopt as a uniform effect.** A single, causal
ad advantage is **not supported** by this observational data: the
signal is plausibly the confounder `total_ads` (exposure is not
randomised). A rollout requires a randomised experiment or a
properly adjusted (causal) analysis first.

**Caveats that must travel with the decision:**
1. Observational cohort → association, not proven causation.
2. No cost data → net economic effect unchecked; gains only.
3. `total_ads` moves from *suspected* to *evidenced* confounder
   (notebook 06): the naive pooled effect does not survive
   stratification.
4. Synthetic benchmark dataset → the *method* is the reusable
   output; absolute revenue figures are illustrative.
5. Low-exposure strata are sparse (expected events < 5) — the
   absence of a detected effect there is read descriptively, not
   as proof of zero effect.

Walk through the full analysis: notebooks 01–04, robustness in
notebook 06, and this summary.
